# L4：Phase 2B 重跑 + Phase 4（L4 對照列）

備案 B（程式本體在 `scripts/`，這裡只是薄殼）。
1. 「執行階段 → 變更執行階段類型」選 **L4**
2. 先記下右上角「資源」面板的**可用運算單元**（開工前餘額）
3. 「執行階段 → 全部執行」，第一次會問是否信任 GitHub 來源，選「仍要執行」
4. 跑完把「結果 JSON」格整段輸出貼回 Claude Code，連同結束後的運算單元餘額
5. 下載 tar.gz 之後執行最後一格釋放 VM（忘了也會在 40 分鐘後被 watchdog 釋放）

In [ ]:
# @title 0. 自毀保險 + 取得腳本（先選好 L4 再執行）
# 40 分鐘後自動 runtime.unassign() 釋放 VM，防止忘記關、手機斷線時一直燒單位。
import threading, time, subprocess
from google.colab import runtime
WATCHDOG_MIN = 40
def _bomb():
    time.sleep(WATCHDOG_MIN * 60); print("[watchdog] 時間到，釋放 runtime", flush=True); runtime.unassign()
threading.Thread(target=_bomb, daemon=True).start()
T0 = time.time()
BRANCH = "claude/keen-turing-d5s729"  # 合併到 main 之後改成 "main"
subprocess.run(f"rm -rf /content/colabpoc && git clone -q --depth 1 -b {BRANCH} https://github.com/clarencechien/colabpoc /content/colabpoc", shell=True, check=True)
%cd /content/colabpoc
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"watchdog armed: {WATCHDOG_MIN} min")

In [ ]:
# @title Phase 0：GPU 探測
!python scripts/phase0_probe.py

In [ ]:
# @title Phase 2B：ZONOS3-Mini（transformers）on L4
!python scripts/phase2_zonos_tts.py --paths B --out /content/out/phase2 2>&1 | tail -c 3000

In [ ]:
# @title Phase 4：Blender（L4）
!python scripts/phase4_blender.py --devices OPTIX,CUDA,CPU 2>&1 | tail -c 3000

In [ ]:
# @title 結果 JSON（整格輸出貼回 Claude Code）
import glob, json
for pat in ['/content/out/*/*.json']:
    for f in sorted(glob.glob(pat)):
        print("=====", f); print(open(f).read())
print(f"wall_min={(time.time()-T0)/60:.1f}")

In [ ]:
# @title 打包 wav / png / 結果，下載後釋放 runtime
!cd /content && tar czf l4_out.tar.gz out && ls -la /content/l4_out.tar.gz
from google.colab import files
files.download("/content/l4_out.tar.gz")

In [ ]:
# @title 最後：確認下載完成後再執行，立刻釋放 VM
runtime.unassign()